In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import os

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

BASE_DIR             = "/home/23dcs510/Smart-Recommendation-System/brain/"

FILTERED_VIS_PATH    = BASE_DIR + "filtered_visual_embeddings.npy"
FILTERED_POI_PATH    = BASE_DIR + "filtered_poi_ids.npy"
FILTERED_PID_PATH    = BASE_DIR + "filtered_photo_ids.npy"
MEMBERSHIP_PATH      = BASE_DIR + "poi_fuzzy_memberships.npy"
CLUSTER_IDS_PATH     = BASE_DIR + "poi_cluster_ids.npy"
CSV_PATH             = BASE_DIR + "usa_europe_geotagged.csv"
OUTPUT_DIR           = BASE_DIR

VISUAL_DIM      = 256
MEMBERSHIP_DIM  = 6
USER_DIM        = VISUAL_DIM + MEMBERSHIP_DIM   # 262
POI_DIM         = VISUAL_DIM + MEMBERSHIP_DIM   # 262
FUSION_DIM      = USER_DIM + POI_DIM            # 524

BATCH_SIZE      = 2048
EPOCHS          = 20
LR              = 1e-3
WEIGHT_DECAY    = 1e-4
EARLY_STOP_PAT  = 4
NEG_SAMPLES     = 4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Base dir: {BASE_DIR}")

Device: cuda
Base dir: /home/23dcs510/Smart-Recommendation-System/brain/


In [3]:
print("Loading embeddings...")
filtered_vis  = np.load(FILTERED_VIS_PATH).astype(np.float32)
filtered_pois = np.load(FILTERED_POI_PATH)
filtered_pids = np.load(FILTERED_PID_PATH)
memberships   = np.load(MEMBERSHIP_PATH).astype(np.float32)  # (169845, 6)
unique_pois   = np.load(CLUSTER_IDS_PATH)

print("Loading CSV...")
df            = pd.read_csv(CSV_PATH)
df['photoid'] = df['photoid'].astype(int)
df['uid']     = df['uid'].astype(str)

# Build photo_id → uid mapping
pid_to_uid    = df.set_index('photoid')['uid'].to_dict()

# Build poi_id → membership vector
poi_to_membership = {
    poi: memberships[idx]
    for idx, poi in enumerate(unique_pois)
}

# Build poi_id → mean visual embedding
print("Building POI visual embeddings...")
poi_to_vis_embs = {}
for i, poi in enumerate(filtered_pois):
    poi_to_vis_embs.setdefault(poi, []).append(filtered_vis[i])

poi_visual_emb = {
    poi: np.mean(embs, axis=0)
    for poi, embs in tqdm(poi_to_vis_embs.items(), desc="Averaging POI embeddings")
}

valid_pois = np.array([p for p in unique_pois if p in poi_visual_emb])
print(f"Valid POIs with visual embeddings: {len(valid_pois):,}")

print(f"\nFiltered visual  : {filtered_vis.shape}")
print(f"Memberships      : {memberships.shape}")
print(f"CSV rows         : {len(df):,}")

Loading embeddings...
Loading CSV...
Building POI visual embeddings...


Averaging POI embeddings: 100%|██████████████████████████████████████████████| 169845/169845 [00:04<00:00, 42024.62it/s]


Valid POIs with visual embeddings: 169,845

Filtered visual  : (767210, 256)
Memberships      : (169845, 6)
CSV rows         : 7,672,417


In [4]:
"""
User Visual DNA:
- Group all filtered photos by user (uid)
- Average their visual embeddings → user visual profile (256-dim)
- Average their POI memberships → user aesthetic profile (6-dim)
- Concatenate → user embedding (262-dim)
"""

print("Building user profiles...")

# Map filtered photos to users
uid_to_vis_embs   = {}
uid_to_memberships = {}

for i, pid in enumerate(tqdm(filtered_pids, desc="Grouping by user")):
    uid = pid_to_uid.get(int(pid), None)
    if uid is None: continue

    poi = filtered_pois[i]
    if poi not in poi_to_membership: continue

    uid_to_vis_embs.setdefault(uid, []).append(filtered_vis[i])
    uid_to_memberships.setdefault(uid, []).append(poi_to_membership[poi])

# Average to get user embeddings
user_ids      = []
user_vis_embs = []
user_mem_embs = []

for uid, vis_list in tqdm(uid_to_vis_embs.items(), desc="Averaging user embeddings"):
    if len(vis_list) < 2: continue  # skip users with too few photos
    user_ids.append(uid)
    user_vis_embs.append(np.mean(vis_list,                    axis=0))
    user_mem_embs.append(np.mean(uid_to_memberships[uid],     axis=0))

user_ids      = np.array(user_ids)
user_vis_embs = np.array(user_vis_embs, dtype=np.float32)
user_mem_embs = np.array(user_mem_embs, dtype=np.float32)
user_embeddings = np.concatenate([user_vis_embs, user_mem_embs], axis=1)  # (N, 262)

print(f"Total users         : {len(user_ids):,}")
print(f"User embeddings     : {user_embeddings.shape}")

Building user profiles...


Averaging user embeddings: 100%|███████████████████████████████████████████████| 32803/32803 [00:01<00:00, 17921.70it/s]


Total users         : 27,071
User embeddings     : (27071, 262)


In [5]:
"""
POI Embedding = mean visual embedding + fuzzy membership vector (262-dim)
"""
poi_ids_list  = []
poi_embeddings_list = []

for poi in tqdm(valid_pois, desc="Building POI embeddings"):
    vis  = poi_visual_emb[poi]                    # (256,)
    mem  = poi_to_membership[poi]                 # (6,)
    emb  = np.concatenate([vis, mem], axis=0)     # (262,)
    poi_ids_list.append(poi)
    poi_embeddings_list.append(emb)

poi_ids_arr  = np.array(poi_ids_list)
poi_emb_arr  = np.array(poi_embeddings_list, dtype=np.float32)  # (P, 262)

# Build lookup
poi_id_to_idx = {poi: idx for idx, poi in enumerate(poi_ids_arr)}

print(f"POI embeddings: {poi_emb_arr.shape}")

Building POI embeddings: 100%|██████████████████████████████████████████████| 169845/169845 [00:00<00:00, 248317.37it/s]

POI embeddings: (169845, 262)


In [6]:
"""
Positive pairs: user → POIs they actually photographed
Negative pairs: user → random POIs they never visited
Label: 1 = visited, 0 = not visited
"""

print("Building user-POI visit history...")
uid_to_pois = {}
for i, pid in enumerate(filtered_pids):
    uid = pid_to_uid.get(int(pid), None)
    if uid is None: continue
    poi = filtered_pois[i]
    if poi in poi_id_to_idx:
        uid_to_pois.setdefault(uid, set()).add(poi)

# Build user index lookup
user_id_to_idx = {uid: idx for idx, uid in enumerate(user_ids)}

print("Sampling training pairs...")
train_user_idx = []
train_poi_idx  = []
train_labels   = []

all_poi_indices = np.arange(len(poi_ids_arr))

for uid in tqdm(user_ids, desc="Building pairs"):
    visited_pois = uid_to_pois.get(uid, set())
    if not visited_pois: continue

    u_idx = user_id_to_idx[uid]

    for poi in visited_pois:
        if poi not in poi_id_to_idx: continue
        p_idx = poi_id_to_idx[poi]

        # Positive
        train_user_idx.append(u_idx)
        train_poi_idx.append(p_idx)
        train_labels.append(1.0)

        # Negatives
        neg_count = 0
        while neg_count < NEG_SAMPLES:
            neg_idx = np.random.randint(0, len(poi_ids_arr))
            if poi_ids_arr[neg_idx] not in visited_pois:
                train_user_idx.append(u_idx)
                train_poi_idx.append(neg_idx)
                train_labels.append(0.0)
                neg_count += 1

train_user_idx = np.array(train_user_idx)
train_poi_idx  = np.array(train_poi_idx)
train_labels   = np.array(train_labels, dtype=np.float32)

print(f"Total training pairs : {len(train_labels):,}")
print(f"Positive pairs       : {train_labels.sum():,.0f}")
print(f"Negative pairs       : {(1-train_labels).sum():,.0f}")

Building user-POI visit history...
Sampling training pairs...


Building pairs: 100%|███████████████████████████████████████████████████████████| 27071/27071 [00:04<00:00, 6323.73it/s]


Total training pairs : 1,211,490
Positive pairs       : 242,298
Negative pairs       : 969,192


In [7]:
class UserPOIDataset(Dataset):
    def __init__(self, user_embs, poi_embs, user_idx, poi_idx, labels):
        self.user_embs = torch.tensor(user_embs, dtype=torch.float32)
        self.poi_embs  = torch.tensor(poi_embs,  dtype=torch.float32)
        self.user_idx  = user_idx
        self.poi_idx   = poi_idx
        self.labels    = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return (
            self.user_embs[self.user_idx[i]],
            self.poi_embs[self.poi_idx[i]],
            self.labels[i]
        )

# 80/20 split
n          = len(train_labels)
val_size   = int(0.2 * n)
train_size = n - val_size
indices    = np.random.permutation(n)
train_idx  = indices[:train_size]
val_idx    = indices[train_size:]

train_ds = UserPOIDataset(
    user_embeddings, poi_emb_arr,
    train_user_idx[train_idx],
    train_poi_idx[train_idx],
    train_labels[train_idx]
)
val_ds = UserPOIDataset(
    user_embeddings, poi_emb_arr,
    train_user_idx[val_idx],
    train_poi_idx[val_idx],
    train_labels[val_idx]
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"Train pairs: {train_size:,} | Val pairs: {val_size:,}")
print(f"Steps/epoch: {len(train_loader)}")

Train pairs: 969,192 | Val pairs: 242,298
Steps/epoch: 474


In [8]:
class NeuralInteractionLayer(nn.Module):
    """
    Fuses user and POI embeddings to predict visit probability.
    Uses element-wise product (interaction) + concatenation
    to capture both direct and cross-feature interactions.
    Sigmoid output → visit probability in [0,1].
    """
    def __init__(self, user_dim=262, poi_dim=262):
        super().__init__()

        # Individual towers
        self.user_tower = nn.Sequential(
            nn.Linear(user_dim, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.poi_tower = nn.Sequential(
            nn.Linear(poi_dim, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        # Interaction layer: concat + element-wise product
        # Input: [user(128), poi(128), interaction(128)] = 384
        self.interaction = nn.Sequential(
            nn.Linear(128 * 3, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, user_emb, poi_emb):
        u   = self.user_tower(user_emb)     # (B, 128)
        p   = self.poi_tower(poi_emb)       # (B, 128)
        ip  = u * p                         # element-wise interaction (B, 128)
        x   = torch.cat([u, p, ip], dim=1) # (B, 384)
        return self.interaction(x).squeeze(-1)  # (B,)

model     = NeuralInteractionLayer().to(DEVICE)
criterion = nn.BCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {total_params:,}")
print(model)

Model parameters: 208,385
NeuralInteractionLayer(
  (user_tower): Sequential(
    (0): Linear(in_features=262, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.2, inplace=False)
  )
  (poi_tower): Sequential(
    (0): Linear(in_features=262, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.2, inplace=False)
  )
  (interaction): Sequential(
    (0): Linear(in_features=384, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): GELU(approximate='none

In [9]:
best_val_loss  = float('inf')
patience_count = 0
best_weights   = None

for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss = 0
    train_correct = 0
    train_total   = 0

    for user_emb, poi_emb, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        user_emb = user_emb.to(DEVICE)
        poi_emb  = poi_emb.to(DEVICE)
        labels   = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        preds = model(user_emb, poi_emb)
        loss  = criterion(preds, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss    += loss.item()
        train_correct += ((preds > 0.5) == labels.bool()).sum().item()
        train_total   += len(labels)

    # Validate
    model.eval()
    val_loss    = 0
    val_correct = 0
    val_total   = 0

    with torch.no_grad():
        for user_emb, poi_emb, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]  "):
            user_emb = user_emb.to(DEVICE)
            poi_emb  = poi_emb.to(DEVICE)
            labels   = labels.to(DEVICE)
            preds    = model(user_emb, poi_emb)
            loss     = criterion(preds, labels)
            val_loss    += loss.item()
            val_correct += ((preds > 0.5) == labels.bool()).sum().item()
            val_total   += len(labels)

    avg_train     = train_loss / len(train_loader)
    avg_val       = val_loss   / len(val_loader)
    train_acc     = train_correct / train_total * 100
    val_acc       = val_correct   / val_total   * 100
    scheduler.step()

    print(f"Epoch {epoch+1:02d} | Train Loss: {avg_train:.4f} Acc: {train_acc:.1f}% | Val Loss: {avg_val:.4f} Acc: {val_acc:.1f}% | LR: {scheduler.get_last_lr()[0]:.6f}")

    if avg_val < best_val_loss:
        best_val_loss  = avg_val
        best_weights   = {k: v.clone() for k, v in model.state_dict().items()}
        patience_count = 0
        print("  ✓ Best model saved.")
    else:
        patience_count += 1
        print(f"  No improvement ({patience_count}/{EARLY_STOP_PAT})")
        if patience_count >= EARLY_STOP_PAT:
            print("Early stopping triggered.")
            break

model.load_state_dict(best_weights)
print(f"\nBest Val Loss: {best_val_loss:.4f}")

Epoch 1/20 [Val]  : 100%|█████████████████████████████████████████████████████████████| 119/119 [00:03<00:00, 34.71it/s]


Epoch 01 | Train Loss: 0.3790 Acc: 83.1% | Val Loss: 0.3543 Acc: 84.2% | LR: 0.000994
  ✓ Best model saved.


Epoch 2/20 [Val]  : 100%|█████████████████████████████████████████████████████████████| 119/119 [00:03<00:00, 36.88it/s]


Epoch 02 | Train Loss: 0.3580 Acc: 84.2% | Val Loss: 0.3467 Acc: 84.6% | LR: 0.000976
  ✓ Best model saved.


Epoch 3/20 [Val]  : 100%|█████████████████████████████████████████████████████████████| 119/119 [00:03<00:00, 34.92it/s]


...

Epoch 18 | Train Loss: 0.3299 Acc: 85.4% | Val Loss: 0.3229 Acc: 85.7% | LR: 0.000034
  ✓ Best model saved.


Epoch 19/20 [Val]  : 100%|████████████████████████████████████████████████████████████| 119/119 [00:03<00:00, 34.29it/s]


Epoch 19 | Train Loss: 0.3296 Acc: 85.5% | Val Loss: 0.3228 Acc: 85.7% | LR: 0.000016
  ✓ Best model saved.


Epoch 20/20 [Val]  : 100%|████████████████████████████████████████████████████████████| 119/119 [00:03<00:00, 32.76it/s]

Epoch 20 | Train Loss: 0.3290 Acc: 85.5% | Val Loss: 0.3226 Acc: 85.8% | LR: 0.000010
  ✓ Best model saved.

Best Val Loss: 0.3226


In [10]:
# Save model
MODEL_PATH = os.path.join(OUTPUT_DIR, "neural_interaction_model.pth")
torch.save(model.state_dict(), MODEL_PATH)
print(f"Model saved → {MODEL_PATH}")

# Save user & POI embeddings for inference
np.save(os.path.join(OUTPUT_DIR, "user_embeddings.npy"),    user_embeddings)
np.save(os.path.join(OUTPUT_DIR, "user_ids.npy"),           user_ids)
np.save(os.path.join(OUTPUT_DIR, "poi_embeddings_262.npy"), poi_emb_arr)
np.save(os.path.join(OUTPUT_DIR, "poi_ids_262.npy"),        poi_ids_arr)

print(f"User embeddings : {user_embeddings.shape}")
print(f"POI embeddings  : {poi_emb_arr.shape}")
print("NB10 Complete. Ready for NB11.")

Model saved → /home/23dcs510/Smart-Recommendation-System/brain/neural_interaction_model.pth
User embeddings : (27071, 262)
POI embeddings  : (169845, 262)
NB10 Complete. Ready for NB11.
